In [37]:
# ============================================================
# IMPORTS STANDARDS
# ============================================================

from pathlib import Path
from urllib.parse import urljoin, urlparse, urldefrag
from datetime import datetime, timezone

import re
import time
import shutil
import hashlib
import importlib
import subprocess
import sys
import math
import tempfile


# ============================================================
# INSTALLATION AUTOMATIQUE DES PACKAGES EXTERNES
# ============================================================

def install_and_import(package_name, import_name=None):
    """
    Installe un package s'il n'est pas présent, puis l'importe.

    package_name : nom utilisé par pip
    import_name  : nom utilisé dans import Python
    """

    if import_name is None:
        import_name = package_name

    try:
        module = importlib.import_module(import_name)
        print(f"[OK] {package_name} déjà installé")

    except ImportError:
        print(f"[INSTALLATION] {package_name}")

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            package_name
        ])

        module = importlib.import_module(import_name)

        print(f"[OK] {package_name} installé")

    return module


# ============================================================
# IMPORTS EXTERNES ROBUSTES
# ============================================================

requests = install_and_import("requests")

pd = install_and_import("pandas")

BeautifulSoup = install_and_import(
    "beautifulsoup4",
    "bs4"
).BeautifulSoup

# API / librairie de recherche web DuckDuckGo
DDGS = install_and_import(
    "ddgs",
    "ddgs"
).DDGS

[OK] requests déjà installé
[OK] pandas déjà installé
[OK] beautifulsoup4 déjà installé
[OK] ddgs déjà installé


# Pipeline robuste de collecte et d’ingestion documentaire ESG

## Objectif de cette première phase

Cette première étape du projet vise à construire un pipeline automatisé capable de :
- découvrir,
- collecter,
- valider,
- scorer,
- sélectionner,
- puis ingérer des documents ESG et réglementaires à partir de multiples sources documentaires.

L’objectif est de constituer une base documentaire ESG structurée, traçable et exploitable pour les étapes ultérieures :
- extraction d’informations ESG,
- construction d’indicateurs,
- scoring ESG,
- analyse de robustesse,
- et recherche quantitative en investissement durable.

---

## Fonctionnalités principales du pipeline

Le pipeline a été conçu pour être robuste, modulaire et extensible.

Il permet notamment :

### 1. Gestion de plusieurs modes d’entrée
Le système accepte différents types d’entrée :
- noms d’entreprises ;
- URLs PDF fournies manuellement ;
- fichiers PDF locaux uploadés.

---

### 2. Discovery documentaire automatisée
Le pipeline combine :
- exploration directe des sites corporate ;
- et recherche web complémentaire via DuckDuckGo Search (DDGS).

Cette approche améliore significativement la couverture documentaire ESG.

---

### 3. Validation et contrôle qualité
Chaque document détecté est :
- validé techniquement ;
- normalisé ;
- dédupliqué ;
- puis enregistré avec ses métadonnées et logs associés.

---

### 4. Scoring et sélection robuste
Les documents candidats sont évalués automatiquement selon :
- leur cohérence documentaire ;
- la présence de mots-clés ESG ;
- l’année fiscale ;
- le type de rapport attendu ;
- et plusieurs critères de pertinence.

Le pipeline distingue ensuite :
- les documents automatiquement sélectionnés ;
- les documents nécessitant une revue manuelle ;
- et les documents probablement absents.

---

### 5. Architecture exploitable pour les étapes futures
Le pipeline produit des sorties structurées et auditables permettant une intégration future avec :
- des outils d’extraction NLP ;
- des systèmes de scoring ESG ;
- une interface graphique ;
- ou des pipelines de recherche quantitative plus avancés.

## 1. Configuration de l’environnement

Cette cellule, configure les dossiers de travail et définit les paramètres globaux utilisés pour la collecte, le stockage et le suivi des documents ESG. On définit ici l’architecture locale du projet. Les documents téléchargés sont stockés dans `raw`, tandis que les fichiers de suivi, de logs et de métadonnées sont centralisés dans `metadata` et `logs`.

In [38]:
BASE_DIR = Path("esg_data")
RAW_DIR = BASE_DIR / "raw"
LOG_DIR = BASE_DIR / "logs"
META_DIR = BASE_DIR / "metadata"

for directory in [RAW_DIR, LOG_DIR, META_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

METADATA_PATH = META_DIR / "documents_metadata.csv"
LOG_PATH = LOG_DIR / "ingestion_log.csv"
DISCOVERY_PATH = META_DIR / "discovery_results.csv"
PDF_INDEX_PATH = META_DIR / "company_pdf_index.csv"
SELECTION_PATH = META_DIR / "selected_documents.csv"


## 2. Paramètres de requête web

Ces paramètres encadrent les téléchargements : identification du script auprès des serveurs, délai maximal d’attente et taille maximale autorisée pour un fichier PDF.

In [39]:
REQUEST_HEADERS = {
    "User-Agent": "Mozilla/5.0 ESG Research Bot; educational research use"
}

REQUEST_TIMEOUT = 25
MAX_DOWNLOAD_MB = 80


## 4. Schémas des fichiers de sortie

Cette cellule définit les colonnes attendues pour chaque fichier de suivi. Cela garantit une structure stable entre les différentes exécutions du pipeline.

In [40]:
# ============================================================
# 2. SCHÉMAS
# ============================================================

LOG_COLUMNS = [
    "input_mode", "company", "ticker", "isin", "jurisdiction",
    "strate", "doc_type", "doc_subtype", "fiscal_year",
    "period_type", "source_url", "local_input_path",
    "status", "error_type", "error_message", "retrieval_date"
]

METADATA_COLUMNS = [
    "input_mode", "company", "ticker", "isin", "jurisdiction",
    "strate", "doc_type", "doc_subtype", "fiscal_year",
    "period_type", "source_url", "local_path", "sha256",
    "file_size_bytes", "retrieval_date", "status"
]

PDF_INDEX_COLUMNS = [
    "company", "ticker", "isin", "jurisdiction",
    "source_page", "candidate_pdf_url", "anchor_text",
    "valid_pdf", "retrieval_date"
]

DISCOVERY_COLUMNS = [
    "company", "ticker", "isin", "jurisdiction",
    "strate", "doc_type", "doc_subtype", "fiscal_year",
    "period_type", "expected_query", "source_page", "candidate_pdf_url",
    "anchor_text", "valid_pdf", "score", "confidence_level",
    "estimated_probability", "discovery_status", "selection_reason",
    "retrieval_date"
]

SELECTION_COLUMNS = [
    "company", "ticker", "isin", "jurisdiction",
    "strate", "doc_type", "doc_subtype", "fiscal_year",
    "period_type", "expected_query", "selected_url", "score",
    "confidence_level", "estimated_probability", "selection_status",
    "selection_reason", "retrieval_date"
]



## 5. Référentiel documentaire ESG

Le référentiel documentaire décrit les types de documents recherchés.  
Chaque document est associé à une strate de collecte, une catégorie, une périodicité et une requête-type utilisée pour la recherche automatique.

In [41]:
# ============================================================
# 3. RÉFÉRENTIEL DOCUMENTAIRE
# ============================================================

DOCUMENT_REGISTRY = [
    {"strate": 1, "doc_type": "regulatory", "doc_subtype": "annual_report_urd", "period_type": "FY",
     "query_template": "{company} annual report universal registration document {year} PDF"},

    {"strate": 1, "doc_type": "regulatory", "doc_subtype": "sustainability_statement_csrd_esrs", "period_type": "FY",
     "query_template": "{company} sustainability statement ESRS CSRD {year} PDF"},

    {"strate": 1, "doc_type": "regulatory", "doc_subtype": "climate_report_tcfd_transition_plan", "period_type": "FY",
     "query_template": "{company} climate report TCFD transition plan {year} PDF"},

    {"strate": 1, "doc_type": "regulatory", "doc_subtype": "vigilance_plan", "period_type": "FY",
     "query_template": "{company} plan de vigilance {year} PDF"},

    {"strate": 1, "doc_type": "regulatory", "doc_subtype": "half_year_financial_report", "period_type": "H1",
     "query_template": "{company} half year financial report {year} PDF"},

    {"strate": 2, "doc_type": "policy", "doc_subtype": "code_of_conduct", "period_type": "perpetual",
     "query_template": "{company} code of conduct PDF"},

    {"strate": 2, "doc_type": "policy", "doc_subtype": "anti_corruption_policy", "period_type": "perpetual",
     "query_template": "{company} anti corruption policy PDF"},

    {"strate": 2, "doc_type": "policy", "doc_subtype": "human_rights_policy", "period_type": "perpetual",
     "query_template": "{company} human rights policy PDF"},

    {"strate": 2, "doc_type": "policy", "doc_subtype": "dei_policy", "period_type": "perpetual",
     "query_template": "{company} diversity equity inclusion policy PDF"},

    {"strate": 2, "doc_type": "policy", "doc_subtype": "environmental_policy", "period_type": "perpetual",
     "query_template": "{company} environmental policy climate biodiversity water PDF"},

    {"strate": 2, "doc_type": "policy", "doc_subtype": "supplier_code_of_conduct", "period_type": "perpetual",
     "query_template": "{company} supplier code of conduct responsible purchasing PDF"},

    {"strate": 3, "doc_type": "corporate_communication", "doc_subtype": "investor_presentation", "period_type": "adhoc",
     "query_template": "{company} investor presentation ESG capital markets day {year} PDF"},

    {"strate": 3, "doc_type": "corporate_communication", "doc_subtype": "agm_minutes_resolutions", "period_type": "adhoc",
     "query_template": "{company} annual general meeting resolutions {year} PDF"},

    {"strate": 4, "doc_type": "external_source", "doc_subtype": "cdp_response", "period_type": "FY",
     "query_template": "{company} CDP climate change response {year} PDF"},

    {"strate": 4, "doc_type": "external_source", "doc_subtype": "sbti_validation", "period_type": "adhoc",
     "query_template": "{company} SBTi validated targets"},

    {"strate": 4, "doc_type": "external_source", "doc_subtype": "assurance_report", "period_type": "FY",
     "query_template": "{company} independent assurance report sustainability {year} PDF"}
]



## 6. Référentiel des entreprises

Cette cellule contient un référentiel initial d’émetteurs utilisé pour accélérer et fiabiliser la phase de discovery documentaire.

Pour les entreprises présentes dans ce référentiel, le pipeline dispose directement d’informations structurées telles que :
- le nom de l’entreprise,
- le ticker,
- l’ISIN,
- la juridiction,
- ainsi que les domaines web officiels à explorer.

Cela permet :
- de limiter les recherches à des sources corporate fiables,
- de réduire le bruit documentaire,
- d’améliorer la précision du scoring des documents candidats,
- et d’accélérer la collecte.

Cependant, le pipeline reste volontairement générique et extensible.

Lorsqu’une entreprise n’est pas présente dans le référentiel, une procédure de normalisation et d’inférence est appliquée automatiquement :
- génération heuristique de domaines probables à partir du nom de l’entreprise,
- exploration web complémentaire via moteurs de recherche,
- puis scoring des documents détectés selon leur pertinence documentaire.

Le référentiel agit donc comme une couche d’optimisation et non comme une dépendance structurelle du pipeline.

In [42]:
# ============================================================
# 4. RÉFÉRENTIEL ENTREPRISES
# ============================================================

EMITTERS = [
    {"company": "TotalEnergies", "ticker": "TTE", "isin": "FR0000120271", "jurisdiction": "France", "official_domains": ["totalenergies.com"]},
    {"company": "Schneider Electric", "ticker": "SU", "isin": "FR0000121972", "jurisdiction": "France", "official_domains": ["se.com", "schneider-electric.com"]},
    {"company": "LVMH", "ticker": "MC", "isin": "FR0000121014", "jurisdiction": "France", "official_domains": ["lvmh.com"]},
    {"company": "Air Liquide", "ticker": "AI", "isin": "FR0000120073", "jurisdiction": "France", "official_domains": ["airliquide.com"]},
    {"company": "Sanofi", "ticker": "SAN", "isin": "FR0000120578", "jurisdiction": "France", "official_domains": ["sanofi.com"]},
    {"company": "Airbus", "ticker": "AIR", "isin": "NL0000235190", "jurisdiction": "Netherlands", "official_domains": ["airbus.com"]},
    {"company": "Safran", "ticker": "SAF", "isin": "FR0000073272", "jurisdiction": "France", "official_domains": ["safran-group.com"]},
    {"company": "BNP Paribas", "ticker": "BNP", "isin": "FR0000131104", "jurisdiction": "France", "official_domains": ["bnpparibas.com"]},
    {"company": "L'Oreal", "ticker": "OR", "isin": "FR0000120321", "jurisdiction": "France", "official_domains": ["loreal.com"]},
    {"company": "AXA", "ticker": "CS", "isin": "FR0000120628", "jurisdiction": "France", "official_domains": ["axa.com"]},

    {"company": "Vinci", "ticker": "DG", "isin": "FR0000125486", "jurisdiction": "France", "official_domains": ["vinci.com"]},
    {"company": "EssilorLuxottica", "ticker": "EL", "isin": "FR0000121667", "jurisdiction": "France", "official_domains": ["essilorluxottica.com"]},
    {"company": "Hermes International", "ticker": "RMS", "isin": "FR0000052292", "jurisdiction": "France", "official_domains": ["hermes.com", "finance.hermes.com"]},
    {"company": "Engie", "ticker": "ENGI", "isin": "FR0010208488", "jurisdiction": "France", "official_domains": ["engie.com"]},
    {"company": "Danone", "ticker": "BN", "isin": "FR0000120644", "jurisdiction": "France", "official_domains": ["danone.com"]},
    {"company": "Societe Generale", "ticker": "GLE", "isin": "FR0000130809", "jurisdiction": "France", "official_domains": ["societegenerale.com"]},
    {"company": "Legrand", "ticker": "LR", "isin": "FR0010307819", "jurisdiction": "France", "official_domains": ["legrandgroup.com", "legrand.com"]},
    {"company": "Saint-Gobain", "ticker": "SGO", "isin": "FR0000125007", "jurisdiction": "France", "official_domains": ["saint-gobain.com"]},
    {"company": "Orange", "ticker": "ORA", "isin": "FR0000133308", "jurisdiction": "France", "official_domains": ["orange.com"]},
    {"company": "Thales", "ticker": "HO", "isin": "FR0000121329", "jurisdiction": "France", "official_domains": ["thalesgroup.com"]},

    {"company": "Veolia", "ticker": "VIE", "isin": "FR0000124141", "jurisdiction": "France", "official_domains": ["veolia.com"]},
    {"company": "Michelin", "ticker": "ML", "isin": "FR001400AJ45", "jurisdiction": "France", "official_domains": ["michelin.com"]},
    {"company": "Kering", "ticker": "KER", "isin": "FR0000121485", "jurisdiction": "France", "official_domains": ["kering.com"]},
    {"company": "ArcelorMittal", "ticker": "MT", "isin": "LU1598757687", "jurisdiction": "Luxembourg", "official_domains": ["arcelormittal.com"]},
    {"company": "STMicroelectronics", "ticker": "STMPA", "isin": "NL0000226223", "jurisdiction": "Netherlands", "official_domains": ["st.com"]},

    {"company": "Accor", "ticker": "AC", "isin": "FR0000120404", "jurisdiction": "France", "official_domains": ["group.accor.com", "accor.com"]},
    {"company": "Bouygues", "ticker": "EN", "isin": "FR0000120503", "jurisdiction": "France", "official_domains": ["bouygues.com"]},
    {"company": "Bureau Veritas", "ticker": "BVI", "isin": "FR0006174348", "jurisdiction": "France", "official_domains": ["bureauveritas.com"]},
    {"company": "Capgemini", "ticker": "CAP", "isin": "FR0000125338", "jurisdiction": "France", "official_domains": ["capgemini.com"]},
    {"company": "Carrefour", "ticker": "CA", "isin": "FR0000120172", "jurisdiction": "France", "official_domains": ["carrefour.com"]},
    {"company": "Credit Agricole", "ticker": "ACA", "isin": "FR0000045072", "jurisdiction": "France", "official_domains": ["credit-agricole.com"]},
    {"company": "Dassault Systemes", "ticker": "DSY", "isin": "FR0014003TT8", "jurisdiction": "France", "official_domains": ["3ds.com"]},
    {"company": "Edenred", "ticker": "EDEN", "isin": "FR0010908533", "jurisdiction": "France", "official_domains": ["edenred.com"]},
    {"company": "Euronext", "ticker": "ENX", "isin": "NL0006294274", "jurisdiction": "Netherlands", "official_domains": ["euronext.com"]},
    {"company": "Eurofins Scientific", "ticker": "ERF", "isin": "FR0014000MR3", "jurisdiction": "France", "official_domains": ["eurofins.com"]},
    {"company": "Pernod Ricard", "ticker": "RI", "isin": "FR0000120693", "jurisdiction": "France", "official_domains": ["pernod-ricard.com"]},
    {"company": "Publicis Groupe", "ticker": "PUB", "isin": "FR0000130577", "jurisdiction": "France", "official_domains": ["publicisgroupe.com"]},
    {"company": "Renault", "ticker": "RNO", "isin": "FR0000131906", "jurisdiction": "France", "official_domains": ["renaultgroup.com"]},
    {"company": "Stellantis", "ticker": "STLAP", "isin": "NL00150001Q9", "jurisdiction": "Netherlands", "official_domains": ["stellantis.com"]},
    {"company": "Unibail-Rodamco-Westfield", "ticker": "URW", "isin": "FR0013326246", "jurisdiction": "France", "official_domains": ["urw.com"]}
]


## 7. Fonctions utilitaires générales

Ces fonctions standardisent les noms de fichiers, génèrent les timestamps, normalisent les URLs et gèrent l’écriture incrémentale des fichiers CSV.

In [43]:
# ============================================================
# 5. OUTILS GÉNÉRAUX
# ============================================================

def safe_name(text):
    text = str(text).strip()
    text = re.sub(r"[^a-zA-Z0-9_-]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text or "unknown"


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def normalize_url(url):
    url = str(url).strip()
    url, _ = urldefrag(url)
    return url


def append_csv(row, path, columns):
    clean_row = {col: row.get(col, None) for col in columns}
    df = pd.DataFrame([clean_row], columns=columns)

    if path.exists():
        df.to_csv(path, mode="a", header=False, index=False)
    else:
        df.to_csv(path, index=False)


def log_event(row):
    append_csv(row, LOG_PATH, LOG_COLUMNS)


def save_metadata(row):
    append_csv(row, METADATA_PATH, METADATA_COLUMNS)


def save_selection(row):
    append_csv(row, SELECTION_PATH, SELECTION_COLUMNS)


def compute_sha256(file_path):
    sha256 = hashlib.sha256()
    with open(file_path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(block)
    return sha256.hexdigest()


def short_hash(text, n=10):
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()[:n]


def already_exists(sha256):
    if not METADATA_PATH.exists():
        return False

    metadata = pd.read_csv(METADATA_PATH)

    if "sha256" not in metadata.columns:
        return False

    return sha256 in metadata["sha256"].astype(str).values


def extract_years_from_text(text):
    years = re.findall(r"\b(20[0-3][0-9])\b", str(text))
    return sorted(set(int(y) for y in years))


def get_years_for_doc(doc_spec, years):
    if doc_spec["period_type"] == "perpetual":
        return ["current"]
    return years


def build_expected_query(doc_spec, company, year):
    template = doc_spec.get("query_template", "")
    return template.format(company=company, year=year)


def tokenize_query(query):
    tokens = re.findall(r"[a-zA-ZÀ-ÿ0-9]{3,}", str(query).lower())
    stopwords = {
        "pdf", "the", "and", "for", "avec", "des", "les", "une",
        "document", "report", "rapport"
    }
    return [t for t in tokens if t not in stopwords]


## 8. Validation, normalisation et ingestion des documents PDF

Cet ensemble de fonctions constitue le cœur opérationnel du pipeline de collecte documentaire.

La première étape consiste à vérifier la validité technique des fichiers téléchargés afin d’éviter l’ingestion de documents corrompus, incomplets ou non conformes au format PDF attendu.

Le pipeline intègre également une couche de normalisation des entreprises permettant :
- d’harmoniser les informations corporate,
- d’associer automatiquement les métadonnées connues aux émetteurs,
- et de gérer les entreprises absentes du référentiel grâce à des mécanismes d’inférence de domaines web.

Une fois les documents identifiés, le système organise automatiquement leur stockage local selon :
- l’entreprise,
- l’année fiscale,
- la strate documentaire,
- et le type de document.

Le téléchargement des PDFs est réalisé de manière robuste avec :
- gestion des erreurs réseau,
- contrôle de taille,
- gestion des timeouts,
- et validation post-téléchargement.

Enfin, chaque document validé est enregistré dans une base de métadonnées centralisée contenant :
- les informations d’origine du document,
- son empreinte SHA-256,
- son emplacement local,
- ainsi que l’historique des événements de collecte.

Cette architecture permet de garantir :
- la traçabilité complète du pipeline,
- l’élimination des doublons,
- la reproductibilité de la collecte,
- et une structuration exploitable pour les étapes ultérieures d’extraction et de scoring ESG.

In [44]:
# ============================================================
# 6. VALIDATION PDF
# ============================================================

def validate_pdf(file_path):
    file_path = Path(file_path)

    if not file_path.exists():
        return False, "TYPE_2_FILE_MISSING", "File not found"

    size = file_path.stat().st_size

    if size == 0:
        return False, "TYPE_2_EMPTY_FILE", "File is empty"

    if size < 10_000:
        return False, "TYPE_2_TOO_SMALL", f"Suspicious file size: {size}"

    with open(file_path, "rb") as f:
        header = f.read(1024)

    if not header.startswith(b"%PDF-"):
        return False, "TYPE_2_NOT_PDF", "File is not a valid PDF"

    return True, None, None


# ============================================================
# 7. NORMALISATION DES ENTREPRISES
# ============================================================

def infer_domains_from_company_name(company):
    clean = company.lower()
    clean = clean.replace("&", "and")
    clean = re.sub(r"[^a-z0-9]+", "", clean)

    if not clean:
        return []

    return [f"{clean}.com"]


def normalize_emitters(companies, emitters_reference=EMITTERS, allow_domain_guess=True):
    emitter_map = {item["company"].lower(): item for item in emitters_reference}
    normalized = []

    for company in companies:
        key = str(company).lower().strip()

        if key in emitter_map:
            item = emitter_map[key].copy()
            item["reference_status"] = "FOUND_IN_REFERENCE"
            normalized.append(item)
        else:
            domains = infer_domains_from_company_name(company) if allow_domain_guess else []
            normalized.append({
                "company": company,
                "ticker": None,
                "isin": None,
                "jurisdiction": None,
                "official_domains": domains,
                "reference_status": "NOT_IN_REFERENCE_DOMAIN_GUESSED" if domains else "NOT_IN_REFERENCE_NO_DOMAIN"
            })

    return normalized


# ============================================================
# 8. ENREGISTREMENT COMMUN D'UN PDF
# ============================================================

def register_pdf(
    input_mode,
    file_path,
    company="unknown_company",
    ticker=None,
    isin=None,
    jurisdiction=None,
    strate=None,
    doc_type=None,
    doc_subtype="unknown_document",
    fiscal_year="unknown",
    period_type=None,
    source_url=None,
    local_input_path=None
):
    file_path = Path(file_path)
    retrieval_date = now_utc()

    valid, error_type, error_message = validate_pdf(file_path)

    if not valid:
        log_event({
            "input_mode": input_mode,
            "company": company,
            "ticker": ticker,
            "isin": isin,
            "jurisdiction": jurisdiction,
            "strate": strate,
            "doc_type": doc_type,
            "doc_subtype": doc_subtype,
            "fiscal_year": fiscal_year,
            "period_type": period_type,
            "source_url": source_url,
            "local_input_path": local_input_path,
            "status": "FAILED",
            "error_type": error_type,
            "error_message": error_message,
            "retrieval_date": retrieval_date
        })
        return None

    sha256 = compute_sha256(file_path)

    if already_exists(sha256):
        log_event({
            "input_mode": input_mode,
            "company": company,
            "ticker": ticker,
            "isin": isin,
            "jurisdiction": jurisdiction,
            "strate": strate,
            "doc_type": doc_type,
            "doc_subtype": doc_subtype,
            "fiscal_year": fiscal_year,
            "period_type": period_type,
            "source_url": source_url,
            "local_input_path": local_input_path,
            "status": "DUPLICATE",
            "error_type": "TYPE_3_DUPLICATE",
            "error_message": "Document already exists",
            "retrieval_date": retrieval_date
        })
        return None

    metadata_row = {
        "input_mode": input_mode,
        "company": company,
        "ticker": ticker,
        "isin": isin,
        "jurisdiction": jurisdiction,
        "strate": strate,
        "doc_type": doc_type,
        "doc_subtype": doc_subtype,
        "fiscal_year": fiscal_year,
        "period_type": period_type,
        "source_url": source_url,
        "local_path": str(file_path),
        "sha256": sha256,
        "file_size_bytes": file_path.stat().st_size,
        "retrieval_date": retrieval_date,
        "status": "SUCCESS"
    }

    save_metadata(metadata_row)

    log_event({
        "input_mode": input_mode,
        "company": company,
        "ticker": ticker,
        "isin": isin,
        "jurisdiction": jurisdiction,
        "strate": strate,
        "doc_type": doc_type,
        "doc_subtype": doc_subtype,
        "fiscal_year": fiscal_year,
        "period_type": period_type,
        "source_url": source_url,
        "local_input_path": local_input_path,
        "status": "SUCCESS",
        "error_type": None,
        "error_message": None,
        "retrieval_date": retrieval_date
    })

    return metadata_row


# ============================================================
# 9. TÉLÉCHARGEMENT PDF
# ============================================================

def make_output_path(company, fiscal_year, input_mode, doc_subtype, source_url=None, original_filename=None, strate=None):
    if strate is not None:
        base_path = RAW_DIR / safe_name(company) / str(fiscal_year) / f"strate_{strate}"
    else:
        base_path = RAW_DIR / safe_name(company) / str(fiscal_year) / input_mode

    folder = base_path / safe_name(doc_subtype)
    folder.mkdir(parents=True, exist_ok=True)

    if original_filename:
        stem = safe_name(Path(original_filename).stem)
        unique = short_hash(str(original_filename))
    elif source_url:
        parsed_name = Path(urlparse(source_url).path).name
        stem = safe_name(Path(parsed_name).stem) if parsed_name.lower().endswith(".pdf") else "document"
        unique = short_hash(source_url)
    else:
        stem = "document"
        unique = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")

    filename = f"{stem}_{unique}.pdf"
    return folder / filename


def download_pdf(url, output_path, timeout=REQUEST_TIMEOUT, max_mb=MAX_DOWNLOAD_MB):
    url = normalize_url(url)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    max_bytes = max_mb * 1024 * 1024

    try:
        with requests.get(
            url,
            timeout=timeout,
            headers=REQUEST_HEADERS,
            stream=True,
            allow_redirects=True
        ) as response:

            if response.status_code != 200:
                return False, "TYPE_1_HTTP_ERROR", f"Status code: {response.status_code}"

            content_length = response.headers.get("Content-Length")

            if content_length and int(content_length) > max_bytes:
                return False, "TYPE_1_FILE_TOO_LARGE", f"File exceeds {max_mb} MB"

            total = 0

            with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
                tmp_path = Path(tmp.name)

                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if not chunk:
                        continue

                    total += len(chunk)

                    if total > max_bytes:
                        tmp_path.unlink(missing_ok=True)
                        return False, "TYPE_1_FILE_TOO_LARGE", f"File exceeds {max_mb} MB"

                    tmp.write(chunk)

        shutil.move(str(tmp_path), str(output_path))
        return True, None, None

    except requests.exceptions.Timeout:
        return False, "TYPE_1_TIMEOUT", "Request timeout"

    except requests.exceptions.SSLError:
        return False, "TYPE_1_SSL_ERROR", "SSL error"

    except requests.exceptions.RequestException as e:
        return False, "TYPE_1_REQUEST_ERROR", str(e)

    except Exception as e:
        return False, "TYPE_1_UNKNOWN_DOWNLOAD_ERROR", str(e)


def ingest_from_url(
    url,
    company="unknown_company",
    ticker=None,
    isin=None,
    jurisdiction=None,
    strate=None,
    doc_type=None,
    doc_subtype="user_url_pdf",
    fiscal_year="unknown",
    period_type=None,
    input_mode="url"
):
    output_path = make_output_path(
        company=company,
        fiscal_year=fiscal_year,
        input_mode=input_mode,
        doc_subtype=doc_subtype,
        source_url=url,
        strate=strate
    )

    success, error_type, error_message = download_pdf(url, output_path)

    if not success:
        log_event({
            "input_mode": input_mode,
            "company": company,
            "ticker": ticker,
            "isin": isin,
            "jurisdiction": jurisdiction,
            "strate": strate,
            "doc_type": doc_type,
            "doc_subtype": doc_subtype,
            "fiscal_year": fiscal_year,
            "period_type": period_type,
            "source_url": url,
            "local_input_path": None,
            "status": "FAILED",
            "error_type": error_type,
            "error_message": error_message,
            "retrieval_date": now_utc()
        })
        return None

    return register_pdf(
        input_mode=input_mode,
        file_path=output_path,
        company=company,
        ticker=ticker,
        isin=isin,
        jurisdiction=jurisdiction,
        strate=strate,
        doc_type=doc_type,
        doc_subtype=doc_subtype,
        fiscal_year=fiscal_year,
        period_type=period_type,
        source_url=url
    )


def ingest_from_local_pdf(
    file_path,
    company="unknown_company",
    ticker=None,
    isin=None,
    jurisdiction=None,
    strate=None,
    doc_type=None,
    doc_subtype="uploaded_pdf",
    fiscal_year="unknown",
    period_type=None
):
    file_path = Path(file_path)

    output_path = make_output_path(
        company=company,
        fiscal_year=fiscal_year,
        input_mode="local_pdf",
        doc_subtype=doc_subtype,
        original_filename=file_path.name,
        strate=strate
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(file_path, output_path)

    return register_pdf(
        input_mode="local_pdf",
        file_path=output_path,
        company=company,
        ticker=ticker,
        isin=isin,
        jurisdiction=jurisdiction,
        strate=strate,
        doc_type=doc_type,
        doc_subtype=doc_subtype,
        fiscal_year=fiscal_year,
        period_type=period_type,
        source_url=None,
        local_input_path=str(file_path)
    )


## 9. Recherche web via DuckDuckGo (DDGS)

Cette fonction implémente une phase de discovery documentaire complémentaire à l’exploration directe des sites corporate.

L’objectif est d’améliorer la couverture documentaire du pipeline lorsqu’un document :
- n’est pas facilement accessible depuis les pages investisseurs ou ESG officielles,
- possède une architecture web difficile à crawler,
- ou est référencé sur des sources externes spécialisées.

À partir du type de document recherché (`doc_spec`), une requête documentaire standardisée est construite automatiquement en intégrant :
- le nom de l’entreprise,
- l’année fiscale,
- ainsi que des mots-clés métiers associés au document attendu.

Une recherche web est ensuite effectuée via DuckDuckGo Search (DDGS).

Le pipeline conserve prioritairement :
- les URLs PDF explicites,
- les pages fortement documentaires,
- ainsi que les résultats contenant des termes compatibles avec le reporting ESG ou financier.

Les résultats retournés sont normalisés dans un format homogène avec le reste du pipeline afin d’être intégrés directement dans les étapes suivantes :
- validation PDF,
- scoring documentaire,
- sélection robuste,
- puis ingestion finale.

Cette approche permet de rendre la collecte plus robuste et moins dépendante de l’architecture spécifique des sites corporate.

In [45]:
def search_pdf_candidates_with_ddgs(company, doc_spec, year, max_results=8, sleep_seconds=1.5):
    """
    Recherche web complémentaire via DDGS.
    Retourne des URLs candidates au même format que l'index corporate.
    """
    query = doc_spec["query_template"].format(company=company, year=year)

    # On force la recherche PDF sans dépendre uniquement de DuckDuckGo
    if "pdf" not in query.lower():
        query = query + " PDF"

    results = []

    try:
        with DDGS() as ddgs:
            search_results = ddgs.text(
                query,
                max_results=max_results
            )

            for item in search_results:
                href = item.get("href")
                title = item.get("title", "")
                body = item.get("body", "")

                if not href:
                    continue

                combined = f"{href} {title} {body}".lower()

                # On garde prioritairement les PDFs ou pages très documentaires
                if ".pdf" in combined or "annual report" in combined or "sustainability" in combined:
                    results.append({
                        "source_page": "DDGS_SEARCH",
                        "candidate_pdf_url": normalize_url(href),
                        "anchor_text": f"{title} {body}".strip()
                    })

        time.sleep(sleep_seconds)

    except Exception as e:
        print(f"[DDGS] Search failed for {company} / {doc_spec['doc_subtype']} / {year}: {e}")

    return results


## 10. Discovery documentaire et construction d’un index PDF intermédiaire

Cette étape constitue le cœur du processus de discovery documentaire du pipeline ESG.

L’objectif est d’identifier automatiquement un ensemble de documents candidats potentiellement pertinents pour l’analyse ESG et financière des entreprises étudiées, avant toute phase de scoring ou d’ingestion.

Le pipeline repose sur une approche hybride combinant deux sources complémentaires de discovery :

### 1. Exploration directe des sites corporate

Une première phase consiste à explorer automatiquement les domaines officiels des entreprises.

À partir des sites corporate connus, le pipeline génère un ensemble de pages candidates typiquement utilisées pour la publication des documents réglementaires et ESG :
- espaces investisseurs,
- publications financières,
- pages ESG et sustainability,
- rapports réglementés,
- documents corporate,
- rapports annuels,
- pages CSR / responsabilité.

Chaque page est ensuite analysée afin d’extraire les liens pointant vers des fichiers PDF potentiels.

Cette approche permet de privilégier :
- des sources corporate fiables,
- une meilleure qualité documentaire,
- et une réduction du bruit informationnel.

---

### 2. Recherche web complémentaire via DuckDuckGo Search (DDGS)

L’exploration directe des sites corporate peut néanmoins être insuffisante dans certains cas :
- documents difficilement accessibles,
- architecture web complexe,
- PDFs non reliés depuis les pages principales,
- sous-domaines non explorés,
- ou documents référencés sur des sources externes spécialisées.

Pour améliorer la couverture documentaire, le pipeline intègre donc une seconde phase de discovery via DuckDuckGo Search (DDGS).

Des requêtes documentaires standardisées sont générées automatiquement à partir :
- du nom de l’entreprise,
- du type de document recherché,
- de l’année fiscale,
- et de mots-clés ESG ou réglementaires.

Les résultats retournés sont ensuite filtrés afin de conserver prioritairement :
- les URLs PDF explicites,
- les pages fortement documentaires,
- ainsi que les contenus compatibles avec le reporting ESG ou financier.

Cette approche permet d’augmenter significativement la robustesse du pipeline et de réduire sa dépendance à la structure spécifique des sites corporate.

---

### 3. Validation et normalisation des PDFs détectés

Les URLs collectées sont ensuite :
- normalisées,
- dédupliquées,
- puis validées techniquement.

Le pipeline vérifie notamment :
- la disponibilité des URLs,
- la cohérence du type MIME,
- ainsi que la signature PDF du fichier.

Un mécanisme de cache est utilisé afin d’éviter des validations réseau répétées sur les mêmes URLs et d’améliorer les performances globales de la collecte.

---

### 4. Construction d’un index PDF intermédiaire

Tous les documents détectés sont centralisés dans un index documentaire intermédiaire (`pdf_index`).

Cet index joue un rôle central dans l’architecture du pipeline :
- centralisation des candidats documentaires,
- suppression des doublons,
- traçabilité des sources,
- séparation entre discovery et scoring,
- réduction des appels réseau inutiles,
- et possibilité de rejouer les étapes de scoring sans refaire la collecte web.

Chaque entrée de l’index contient notamment :
- l’entreprise associée,
- l’URL candidate,
- la page source,
- le texte d’ancrage,
- le statut de validation PDF,
- ainsi que les métadonnées temporelles de collecte.

Cet index constitue ensuite la base de travail pour les étapes suivantes :
- scoring documentaire,
- sélection robuste des meilleurs documents,
- téléchargement,
- puis ingestion finale dans la base documentaire ESG.

In [46]:
# ============================================================
# 10. DISCOVERY
# ============================================================

def generate_candidate_pages(official_domains):
    pages = []

    paths = [
        "",
        "/investors",
        "/investors/publications",
        "/investors/regulated-information",
        "/finance",
        "/finance/publications",
        "/sustainability",
        "/sustainability/publications",
        "/group/sustainability",
        "/publications",
        "/documents",
        "/reports",
        "/responsibility",
        "/csr",
        "/esg",
        "/en",
        "/en/investors",
        "/en/sustainability",
        "/fr",
        "/fr/investors",
        "/fr/developpement-durable",
    ]

    for domain in official_domains:
        if not domain:
            continue

        domain = str(domain).strip()

        bases = []
        if domain.startswith("http"):
            bases.append(domain.rstrip("/"))
        else:
            bases.append(f"https://www.{domain}".rstrip("/"))
            bases.append(f"https://{domain}".rstrip("/"))

        for base in bases:
            for path in paths:
                pages.append(f"{base}{path}")

    return list(dict.fromkeys(pages))


def extract_pdf_links_from_page(page_url, timeout=12):
    try:
        response = requests.get(
            page_url,
            timeout=timeout,
            headers=REQUEST_HEADERS,
            allow_redirects=True
        )

        if response.status_code != 200:
            return []

        soup = BeautifulSoup(response.text, "html.parser")
        links = []

        for tag in soup.find_all("a", href=True):
            href = tag["href"]
            anchor_text = tag.get_text(" ", strip=True)

            combined = f"{href} {anchor_text}".lower()

            if ".pdf" in combined:
                candidate_url = normalize_url(urljoin(page_url, href))

                links.append({
                    "source_page": page_url,
                    "candidate_pdf_url": candidate_url,
                    "anchor_text": anchor_text
                })

        return links

    except requests.exceptions.RequestException:
        return []


PDF_VALIDATION_CACHE = {}


def is_valid_pdf_url(url, timeout=10):
    url = normalize_url(url)

    if url in PDF_VALIDATION_CACHE:
        return PDF_VALIDATION_CACHE[url]

    try:
        with requests.get(
            url,
            timeout=timeout,
            stream=True,
            headers=REQUEST_HEADERS,
            allow_redirects=True
        ) as response:

            if response.status_code != 200:
                PDF_VALIDATION_CACHE[url] = False
                return False

            content_type = response.headers.get("Content-Type", "").lower()

            if "pdf" in content_type:
                PDF_VALIDATION_CACHE[url] = True
                return True

            first_bytes = next(response.iter_content(chunk_size=5), b"")
            is_pdf = first_bytes == b"%PDF-"

            PDF_VALIDATION_CACHE[url] = is_pdf
            return is_pdf

    except requests.exceptions.RequestException:
        PDF_VALIDATION_CACHE[url] = False
        return False


def build_company_pdf_index(company, emitter, years=None, document_registry=None, use_ddgs=True):
    pages = generate_candidate_pages(emitter.get("official_domains", []))
    raw_links = []

    # 1. Discovery corporate classique
    for page_url in pages:
        raw_links.extend(extract_pdf_links_from_page(page_url))
        time.sleep(0.1)

    # 2. Discovery complémentaire via DDGS
    if use_ddgs and years is not None and document_registry is not None:
        for doc_spec in document_registry:
            for year in get_years_for_doc(doc_spec, years):
                ddgs_links = search_pdf_candidates_with_ddgs(
                    company=company,
                    doc_spec=doc_spec,
                    year=year,
                    max_results=5,
                    sleep_seconds=1.5
                )
                raw_links.extend(ddgs_links)

    if not raw_links:
        return pd.DataFrame(columns=PDF_INDEX_COLUMNS)

    pdf_index = pd.DataFrame(raw_links)
    pdf_index = pdf_index.drop_duplicates(subset=["candidate_pdf_url"]).reset_index(drop=True)

    pdf_index["company"] = company
    pdf_index["ticker"] = emitter.get("ticker")
    pdf_index["isin"] = emitter.get("isin")
    pdf_index["jurisdiction"] = emitter.get("jurisdiction")
    pdf_index["valid_pdf"] = pdf_index["candidate_pdf_url"].apply(is_valid_pdf_url)
    pdf_index["retrieval_date"] = now_utc()

    pdf_index = pdf_index[PDF_INDEX_COLUMNS]

    pdf_index.to_csv(
        PDF_INDEX_PATH,
        mode="a" if PDF_INDEX_PATH.exists() else "w",
        header=not PDF_INDEX_PATH.exists(),
        index=False
    )

    return pdf_index



## 11. Scoring documentaire et évaluation de pertinence des PDFs

Après la phase de discovery, le pipeline dispose généralement d’un grand nombre de documents candidats potentiels.  
Cette étape a pour objectif d’évaluer automatiquement la pertinence de chaque document détecté afin d’identifier les PDFs les plus cohérents avec le document ESG réellement recherché.

Le système de scoring repose sur une logique heuristique multicritère combinant :
- correspondance avec le nom de l’entreprise,
- cohérence de l’année fiscale,
- présence de mots-clés documentaires positifs,
- détection de mots-clés négatifs,
- proximité avec la requête documentaire attendue,
- ainsi que certains indices structurels comme l’extension PDF.

---

### 1. Référentiel de mots-clés documentaires

Chaque type de document ESG ou réglementaire possède son propre ensemble de mots-clés métiers.

Par exemple :
- rapports annuels,
- sustainability statements,
- rapports TCFD,
- plans de vigilance,
- politiques ESG,
- réponses CDP,
- validations SBTi,
- rapports d’assurance.

Ces référentiels permettent d’orienter le scoring vers les documents les plus susceptibles de correspondre à la catégorie recherchée.

Le pipeline intègre également des mots-clés négatifs afin de pénaliser certains faux positifs fréquents :
- communiqués de presse,
- droits de vote,
- présentations investisseurs,
- rapports semestriels,
- documents non pertinents.

---

### 2. Scoring heuristique des documents candidats

Chaque document candidat est transformé en une représentation textuelle combinant :
- l’URL du document,
- le texte d’ancrage,
- ainsi que certains éléments contextuels.

Le pipeline attribue ensuite un score de pertinence selon plusieurs critères :
- présence du nom de l’entreprise,
- exactitude de l’année fiscale,
- présence des mots-clés attendus,
- similarité avec la requête documentaire cible,
- et cohérence globale du document.

Les incohérences détectées peuvent entraîner des pénalités de score :
- mauvaise année,
- absence d’année,
- présence de termes incompatibles avec le document recherché.

---

### 3. Conversion du score en niveau de confiance

Le score brut obtenu est ensuite converti en :
- pseudo-probabilité estimée de pertinence,
- ainsi qu’en niveau de confiance interprétable :
  - HIGH_CONFIDENCE,
  - MEDIUM_CONFIDENCE,
  - LOW_CONFIDENCE,
  - ou REJECTED.

Cette étape permet de rendre le pipeline plus robuste et plus facilement exploitable pour les phases de sélection automatique.

---

### 4. Construction d’une base de résultats scorés

L’ensemble des documents candidats scorés est ensuite regroupé dans une table de discovery enrichie contenant :
- les informations de l’entreprise,
- le type de document recherché,
- l’URL candidate,
- le score obtenu,
- les raisons du scoring,
- ainsi que les métadonnées de collecte.

Même lorsqu’aucun document pertinent n’est trouvé, le pipeline génère des lignes explicites documentant l’absence de résultat.  
Cela garantit :
- une meilleure traçabilité,
- une couverture documentaire mesurable,
- ainsi qu’une auditabilité complète du pipeline.

Cette étape constitue donc le mécanisme central de filtrage intelligent entre la discovery documentaire brute et la sélection finale des documents ESG à ingérer.

In [47]:
# ============================================================
# 11. SCORING
# ============================================================

KEYWORDS_BY_DOC = {
    "annual_report_urd": [
        "annual report", "universal registration document", "registration document", "urd",
        "document d'enregistrement universel", "rapport annuel"
    ],
    "sustainability_statement_csrd_esrs": [
        "sustainability statement", "csrd", "esrs", "sustainability report",
        "rapport de durabilité", "non-financial statement", "dpef",
        "declaration de performance extra-financiere"
    ],
    "climate_report_tcfd_transition_plan": [
        "climate", "tcfd", "transition plan", "climat", "climate report"
    ],
    "vigilance_plan": [
        "vigilance", "devoir de vigilance", "plan de vigilance"
    ],
    "half_year_financial_report": [
        "half-year", "half year", "interim", "semestriel", "half-yearly"
    ],
    "code_of_conduct": [
        "code of conduct", "ethics", "ethique", "code of ethics"
    ],
    "anti_corruption_policy": [
        "anti corruption", "anti-corruption", "bribery", "corruption"
    ],
    "human_rights_policy": [
        "human rights", "droits humains"
    ],
    "dei_policy": [
        "diversity", "equity", "inclusion", "dei", "diversité", "equal opportunities"
    ],
    "environmental_policy": [
        "environment", "environmental", "climate", "biodiversity", "water", "environnement"
    ],
    "supplier_code_of_conduct": [
        "supplier", "purchasing", "achats", "responsible purchasing", "supplier code"
    ],
    "investor_presentation": [
        "investor presentation", "capital markets", "roadshow", "presentation"
    ],
    "agm_minutes_resolutions": [
        "agm", "general meeting", "resolution", "assemblée générale", "shareholders meeting"
    ],
    "cdp_response": [
        "cdp", "climate change"
    ],
    "sbti_validation": [
        "sbti", "science based", "science-based"
    ],
    "assurance_report": [
        "assurance", "limited assurance", "reasonable assurance", "audit", "verification statement"
    ]
}

NEGATIVE_KEYWORDS_BY_DOC = {
    "annual_report_urd": [
        "voting rights", "voting-rights", "press release", "half-year", "presentation"
    ],
    "half_year_financial_report": [
        "annual report", "universal registration document"
    ],
    "investor_presentation": [
        "voting rights", "press release"
    ],
    "agm_minutes_resolutions": [
        "annual report", "sustainability report"
    ]
}


def score_to_probability(score):
    return round(1 / (1 + math.exp(-(score - 10) / 5)), 3)


def confidence_from_score(score, year_match=True):
    if score is None:
        return "NO_CANDIDATE"
    if score >= 18 and year_match:
        return "HIGH_CONFIDENCE"
    if score >= 10:
        return "MEDIUM_CONFIDENCE"
    if score >= 5:
        return "LOW_CONFIDENCE"
    return "REJECTED"


def score_pdf_candidate(pdf_url, anchor_text, company, doc_subtype, year, expected_query=None):
    text = f"{pdf_url} {anchor_text}".lower()
    score = 0
    reasons = []

    company_tokens = re.findall(r"[a-zA-Z0-9]{3,}", company.lower())

    for token in company_tokens:
        if token in text:
            score += 2
            reasons.append(f"company_token:{token}")

    year_match = True
    detected_years = extract_years_from_text(text)

    if year != "current":
        year = int(year)

        if str(year) in text:
            score += 8
            reasons.append("exact_year_match")
            year_match = True
        elif detected_years:
            nearest_gap = min(abs(year - y) for y in detected_years)

            if nearest_gap == 1:
                score -= 3
                reasons.append("near_year_mismatch")
            else:
                score -= 6
                reasons.append("far_year_mismatch")

            year_match = False
        else:
            score -= 4
            reasons.append("year_missing")
            year_match = False

    for keyword in KEYWORDS_BY_DOC.get(doc_subtype, []):
        if keyword in text:
            score += 5
            reasons.append(f"keyword:{keyword}")

    for keyword in NEGATIVE_KEYWORDS_BY_DOC.get(doc_subtype, []):
        if keyword in text:
            score -= 6
            reasons.append(f"negative_keyword:{keyword}")

    if expected_query:
        query_tokens = tokenize_query(expected_query)
        matched_tokens = [token for token in query_tokens if token in text]

        if matched_tokens:
            query_bonus = min(6, len(set(matched_tokens)))
            score += query_bonus
            reasons.append(f"query_token_overlap:{query_bonus}")

    if ".pdf" in text:
        score += 2
        reasons.append("pdf_extension")

    probability = score_to_probability(score)
    confidence = confidence_from_score(score, year_match=year_match)

    return {
        "score": score,
        "confidence_level": confidence,
        "estimated_probability": probability,
        "year_match": year_match,
        "reasons": "; ".join(reasons)
    }


def missing_discovery_row(company, emitter, doc_spec, year, expected_query, status, reason):
    return {
        "company": company,
        "ticker": emitter.get("ticker"),
        "isin": emitter.get("isin"),
        "jurisdiction": emitter.get("jurisdiction"),
        "strate": doc_spec["strate"],
        "doc_type": doc_spec["doc_type"],
        "doc_subtype": doc_spec["doc_subtype"],
        "fiscal_year": year,
        "period_type": doc_spec["period_type"],
        "expected_query": expected_query,
        "source_page": None,
        "candidate_pdf_url": None,
        "anchor_text": None,
        "valid_pdf": False,
        "score": None,
        "confidence_level": "NO_CANDIDATE",
        "estimated_probability": None,
        "discovery_status": status,
        "selection_reason": reason,
        "retrieval_date": now_utc()
    }


def score_pdf_index_against_registry(company, emitter, pdf_index, years, document_registry):
    rows = []

    valid_index = pdf_index[pdf_index["valid_pdf"] == True].copy() if not pdf_index.empty else pd.DataFrame()

    for doc_spec in document_registry:
        for year in get_years_for_doc(doc_spec, years):
            expected_query = build_expected_query(doc_spec, company, year)

            if valid_index.empty:
                rows.append(
                    missing_discovery_row(
                        company=company,
                        emitter=emitter,
                        doc_spec=doc_spec,
                        year=year,
                        expected_query=expected_query,
                        status="NO_VALID_PDF_IN_INDEX",
                        reason="No valid PDF was available in the explored corporate pages."
                    )
                )
                continue

            for _, pdf_row in valid_index.iterrows():
                scoring = score_pdf_candidate(
                    pdf_url=pdf_row["candidate_pdf_url"],
                    anchor_text=pdf_row["anchor_text"],
                    company=company,
                    doc_subtype=doc_spec["doc_subtype"],
                    year=year,
                    expected_query=expected_query
                )

                rows.append({
                    "company": company,
                    "ticker": emitter.get("ticker"),
                    "isin": emitter.get("isin"),
                    "jurisdiction": emitter.get("jurisdiction"),
                    "strate": doc_spec["strate"],
                    "doc_type": doc_spec["doc_type"],
                    "doc_subtype": doc_spec["doc_subtype"],
                    "fiscal_year": year,
                    "period_type": doc_spec["period_type"],
                    "expected_query": expected_query,
                    "source_page": pdf_row["source_page"],
                    "candidate_pdf_url": pdf_row["candidate_pdf_url"],
                    "anchor_text": pdf_row["anchor_text"],
                    "valid_pdf": True,
                    "score": scoring["score"],
                    "confidence_level": scoring["confidence_level"],
                    "estimated_probability": scoring["estimated_probability"],
                    "discovery_status": "SCORED_CANDIDATE",
                    "selection_reason": scoring["reasons"],
                    "retrieval_date": now_utc()
                })

    discovery = pd.DataFrame(rows, columns=DISCOVERY_COLUMNS)

    if not discovery.empty:
        discovery.to_csv(
            DISCOVERY_PATH,
            mode="a" if DISCOVERY_PATH.exists() else "w",
            header=not DISCOVERY_PATH.exists(),
            index=False
        )

    return discovery


## 12. Sélection robuste, orchestration du pipeline et modes d’ingestion

Cette dernière couche du pipeline transforme les résultats de discovery et de scoring en une collecte documentaire exploitable et structurée.

L’objectif est de passer d’un ensemble de documents candidats potentiels à une sélection robuste de documents ESG réellement pertinents, tout en garantissant :
- la traçabilité des décisions,
- la reproductibilité du pipeline,
- et la robustesse de l’ingestion finale.

---

## 1. Sélection robuste des meilleurs documents

Après la phase de scoring, plusieurs documents candidats peuvent exister pour un même :
- émetteur,
- exercice fiscal,
- et type documentaire.

Le pipeline applique alors une logique de sélection robuste visant à conserver uniquement le meilleur candidat documentaire par catégorie.

La sélection repose sur des seuils de confiance configurables :
- un seuil de téléchargement automatique,
- ainsi qu’un seuil intermédiaire nécessitant une revue manuelle.

Trois cas principaux sont alors distingués :
- document automatiquement sélectionné pour ingestion,
- document nécessitant une validation humaine complémentaire,
- ou absence de document suffisamment pertinent.

Cette approche permet :
- de limiter les faux positifs,
- d’éviter les ingestions erronées,
- et de conserver une logique de contrôle qualité explicite.

Toutes les décisions de sélection sont enregistrées dans une table dédiée afin de garantir l’auditabilité complète du pipeline.

---

## 2. Orchestration complète du pipeline documentaire

Le pipeline implémente ensuite une logique d’orchestration unifiée reliant automatiquement :
- la normalisation des entreprises,
- la discovery documentaire,
- le scoring,
- la sélection,
- puis l’ingestion finale des documents retenus.

Cette orchestration permet de lancer une collecte ESG complète à partir d’une simple liste d’entreprises et d’années fiscales.

Le pipeline :
1. normalise les émetteurs,
2. construit un index documentaire par entreprise,
3. score les documents candidats,
4. sélectionne les meilleurs documents,
5. puis télécharge et enregistre automatiquement les PDFs validés.

Cette architecture modulaire facilite :
- la maintenance,
- l’extensibilité,
- la reproductibilité,
- ainsi que l’industrialisation potentielle du pipeline.

---

## 3. Gestion de plusieurs modes d’entrée utilisateur

Le pipeline ne dépend pas exclusivement de la discovery automatique.

En complément de la collecte web, il permet également :
- l’ingestion directe d’URLs fournies par l’utilisateur,
- ainsi que l’import manuel de fichiers PDF locaux.

Cette flexibilité permet :
- d’intégrer des documents obtenus manuellement,
- de compléter certains cas de collecte complexes,
- ou d’enrichir ponctuellement la base documentaire ESG.

Tous les documents importés suivent ensuite exactement les mêmes étapes :
- validation,
- normalisation,
- déduplication,
- enregistrement des métadonnées,
- et traçabilité des événements de collecte.

Le pipeline conserve ainsi une logique unifiée quel que soit le mode d’entrée utilisé.

In [48]:
# ============================================================
# 12. SÉLECTION ROBUSTE
# ============================================================

def select_best_documents(discovery_results, min_score_download=10, min_score_review=5):
    if discovery_results.empty:
        return pd.DataFrame(columns=SELECTION_COLUMNS)

    selected_rows = []
    group_cols = ["company", "fiscal_year", "doc_subtype"]

    for _, group in discovery_results.groupby(group_cols, dropna=False):
        scored = group[group["score"].notna()].copy()

        if scored.empty:
            best = group.iloc[0]

            row = {
                "company": best["company"],
                "ticker": best["ticker"],
                "isin": best["isin"],
                "jurisdiction": best["jurisdiction"],
                "strate": best["strate"],
                "doc_type": best["doc_type"],
                "doc_subtype": best["doc_subtype"],
                "fiscal_year": best["fiscal_year"],
                "period_type": best["period_type"],
                "expected_query": best["expected_query"],
                "selected_url": None,
                "score": None,
                "confidence_level": "NO_CANDIDATE",
                "estimated_probability": None,
                "selection_status": best["discovery_status"],
                "selection_reason": best["selection_reason"],
                "retrieval_date": now_utc()
            }

            selected_rows.append(row)
            save_selection(row)
            continue

        scored = scored.sort_values("score", ascending=False)
        best = scored.iloc[0]

        if best["score"] >= min_score_download:
            status = "SELECTED_FOR_DOWNLOAD"
            reason = "Best candidate passed automatic download threshold."
            selected_url = best["candidate_pdf_url"]

        elif best["score"] >= min_score_review:
            status = "REVIEW_REQUIRED"
            reason = "Candidate exists but confidence is not sufficient for automatic ingestion."
            selected_url = None

        else:
            status = "NO_RELEVANT_DOCUMENT_FOUND"
            reason = "No candidate reached the minimum relevance threshold."
            selected_url = None

        row = {
            "company": best["company"],
            "ticker": best["ticker"],
            "isin": best["isin"],
            "jurisdiction": best["jurisdiction"],
            "strate": best["strate"],
            "doc_type": best["doc_type"],
            "doc_subtype": best["doc_subtype"],
            "fiscal_year": best["fiscal_year"],
            "period_type": best["period_type"],
            "expected_query": best["expected_query"],
            "selected_url": selected_url,
            "score": best["score"],
            "confidence_level": best["confidence_level"],
            "estimated_probability": best["estimated_probability"],
            "selection_status": status,
            "selection_reason": reason,
            "retrieval_date": now_utc()
        }

        selected_rows.append(row)
        save_selection(row)

    return pd.DataFrame(selected_rows, columns=SELECTION_COLUMNS)


# ============================================================
# 13. ORCHESTRATION DISCOVERY → SÉLECTION → INGESTION
# ============================================================

def run_document_discovery_optimized(
    companies,
    years,
    emitters_reference=EMITTERS,
    document_registry=DOCUMENT_REGISTRY,
    allow_domain_guess=True
):
    normalized_emitters = normalize_emitters(
        companies=companies,
        emitters_reference=emitters_reference,
        allow_domain_guess=allow_domain_guess
    )

    all_discovery = []

    for emitter in normalized_emitters:
        company = emitter["company"]

        print(f"\n[DISCOVERY] Company: {company}")
        print(f"[DISCOVERY] Reference status: {emitter.get('reference_status')}")
        print(f"[DISCOVERY] Explored domains: {emitter.get('official_domains', [])}")

        pdf_index = build_company_pdf_index(
            company=company,
            emitter=emitter,
            years=years,
            document_registry=document_registry,
            use_ddgs=True
        )
        
        print(f"[DISCOVERY] Unique PDF candidates found: {len(pdf_index)}")

        discovery = score_pdf_index_against_registry(
            company=company,
            emitter=emitter,
            pdf_index=pdf_index,
            years=years,
            document_registry=document_registry
        )

        all_discovery.append(discovery)

    if not all_discovery:
        return pd.DataFrame(columns=DISCOVERY_COLUMNS)

    return pd.concat(all_discovery, ignore_index=True)


def ingest_company_documents_from_selection(selection_df, emitters_reference=EMITTERS):
    if selection_df.empty:
        return []

    emitter_map = {item["company"]: item for item in emitters_reference}
    results = []

    selected = selection_df[selection_df["selection_status"] == "SELECTED_FOR_DOWNLOAD"].copy()

    for _, row in selected.iterrows():
        company = row["company"]
        emitter = emitter_map.get(company, {})

        result = ingest_from_url(
            url=row["selected_url"],
            company=company,
            ticker=emitter.get("ticker", row.get("ticker")),
            isin=emitter.get("isin", row.get("isin")),
            jurisdiction=emitter.get("jurisdiction", row.get("jurisdiction")),
            strate=row["strate"],
            doc_type=row["doc_type"],
            doc_subtype=row["doc_subtype"],
            fiscal_year=row["fiscal_year"],
            period_type=row["period_type"],
            input_mode="company_name"
        )

        results.append(result)
        time.sleep(0.3)

    return results


def ingest_from_company_names(
    companies,
    years,
    min_score_download=10,
    min_score_review=5,
    allow_domain_guess=True
):
    discovery_results = run_document_discovery_optimized(
        companies=companies,
        years=years,
        allow_domain_guess=allow_domain_guess
    )

    selection_df = select_best_documents(
        discovery_results=discovery_results,
        min_score_download=min_score_download,
        min_score_review=min_score_review
    )

    ingestion_results = ingest_company_documents_from_selection(selection_df)

    return {
        "discovery_results": discovery_results,
        "selection_results": selection_df,
        "ingestion_results": ingestion_results
    }


# ============================================================
# 14. MODES D'ENTRÉE UTILISATEUR
# ============================================================

def ingest_user_urls(urls, company="user_provided", doc_subtype="user_url_pdf", fiscal_year="unknown"):
    results = []

    for url in urls:
        result = ingest_from_url(
            url=url,
            company=company,
            doc_subtype=doc_subtype,
            fiscal_year=fiscal_year,
            input_mode="user_url"
        )
        results.append(result)

    return results


def ingest_user_local_pdfs(file_paths, company="user_provided", doc_subtype="uploaded_pdf", fiscal_year="unknown"):
    results = []

    for file_path in file_paths:
        result = ingest_from_local_pdf(
            file_path=file_path,
            company=company,
            doc_subtype=doc_subtype,
            fiscal_year=fiscal_year
        )
        results.append(result)
        
    return results
    

## 13. Réinitialisation du pipeline de collecte

Cette cellule remet le pipeline dans un état propre avant une nouvelle exécution.

Elle supprime :
- les fichiers de logs et de métadonnées,
- les résultats de discovery et de sélection,
- ainsi que les PDFs précédemment téléchargés dans `raw/`.

Les dossiers devenus vides sont ensuite automatiquement nettoyés.

Cette étape permet de relancer une collecte complète sans interférence avec les exécutions précédentes et garantit une meilleure reproductibilité du pipeline.

In [49]:
# Supprimer les fichiers CSV
for file in [LOG_PATH, METADATA_PATH, DISCOVERY_PATH, PDF_INDEX_PATH, SELECTION_PATH]:
    if file.exists():
        file.unlink()
        print(f"Supprimé : {file}")

# Supprimer tous les PDF téléchargés dans raw/
if RAW_DIR.exists():
    for pdf_file in RAW_DIR.rglob("*.pdf"):
        pdf_file.unlink()
        print(f"PDF supprimé : {pdf_file}")

# Optionnel : supprimer les dossiers vides dans raw/
for folder in sorted(RAW_DIR.rglob("*"), reverse=True):
    if folder.is_dir() and not any(folder.iterdir()):
        folder.rmdir()
        print(f"Dossier vide supprimé : {folder}")

## 14. Exécution principale sur l’univers CAC 40

Cette cellule lance la collecte documentaire principale sur l’ensemble des entreprises du CAC 40 définies dans le référentiel `EMITTERS`.

Le pipeline distingue deux niveaux de documents :
- les documents prioritaires, utilisés pour la collecte principale ;
- les documents secondaires, conservés pour d’éventuels compléments ultérieurs.

Le run principal se concentre ici sur les documents ESG et réglementaires les plus importants :
- rapports annuels / URD,
- sustainability statements,
- rapports climat ou TCFD,
- plans de vigilance,
- rapports d’assurance.

Pour chaque entreprise et chaque année étudiée, le pipeline exécute successivement :
- la discovery documentaire,
- le scoring des PDFs candidats,
- la sélection robuste des meilleurs documents,
- puis l’ingestion automatique des documents suffisamment fiables.

Les résultats imprimés permettent de contrôler rapidement :
- le volume de documents détectés,
- le nombre de documents sélectionnés,
- le nombre de PDFs effectivement ingérés,
- ainsi que la répartition des statuts de sélection.

In [50]:
# ============================================================
# 0. Entreprises CAC 40
# ============================================================

CAC40_COMPANIES = [emitter["company"] for emitter in EMITTERS]


# ============================================================
# 1. Registre principal : documents ESG prioritaires
# ============================================================

CORE_REPORTS_REGISTRY = [
    doc for doc in DOCUMENT_REGISTRY
    if doc["doc_subtype"] in [
        "annual_report_urd",
        "sustainability_statement_csrd_esrs",
        "climate_report_tcfd_transition_plan",
        "vigilance_plan",
        "assurance_report"
    ]
]


# ============================================================
# 2. Registre secondaire : documents complémentaires
# ============================================================

SECONDARY_REPORTS_REGISTRY = [
    doc for doc in DOCUMENT_REGISTRY
    if doc["doc_subtype"] in [
        "code_of_conduct",
        "anti_corruption_policy",
        "human_rights_policy",
        "dei_policy",
        "environmental_policy",
        "supplier_code_of_conduct",
        "investor_presentation",
        "agm_minutes_resolutions",
        "cdp_response",
        "sbti_validation",
        "half_year_financial_report"
    ]
]

# ============================================================
# RUN PRINCIPAL CORRIGÉ — documents importants uniquement
# ============================================================

discovery_cac40_core = run_document_discovery_optimized(
    companies=CAC40_COMPANIES,
    years=[2023, 2024],
    emitters_reference=EMITTERS,
    document_registry=CORE_REPORTS_REGISTRY,
    allow_domain_guess=False
)

selection_cac40_core = select_best_documents(
    discovery_results=discovery_cac40_core,
    min_score_download=12,
    min_score_review=7
)

ingestion_cac40_core = ingest_company_documents_from_selection(
    selection_df=selection_cac40_core,
    emitters_reference=EMITTERS
)

print("===== RUN PRINCIPAL — DOCUMENTS IMPORTANTS =====")
print("Discovery:", discovery_cac40_core.shape)
print("Selection:", selection_cac40_core.shape)
print("Documents ingérés:", len([x for x in ingestion_cac40_core if x is not None]))
print(selection_cac40_core["selection_status"].value_counts(dropna=False))


[DISCOVERY] Company: TotalEnergies
[DISCOVERY] Reference status: FOUND_IN_REFERENCE
[DISCOVERY] Explored domains: ['totalenergies.com']
[DISCOVERY] Unique PDF candidates found: 45

[DISCOVERY] Company: Schneider Electric
[DISCOVERY] Reference status: FOUND_IN_REFERENCE
[DISCOVERY] Explored domains: ['se.com', 'schneider-electric.com']
[DISCOVERY] Unique PDF candidates found: 45

[DISCOVERY] Company: LVMH
[DISCOVERY] Reference status: FOUND_IN_REFERENCE
[DISCOVERY] Explored domains: ['lvmh.com']
[DISCOVERY] Unique PDF candidates found: 35

[DISCOVERY] Company: Air Liquide
[DISCOVERY] Reference status: FOUND_IN_REFERENCE
[DISCOVERY] Explored domains: ['airliquide.com']
[DISCOVERY] Unique PDF candidates found: 112

[DISCOVERY] Company: Sanofi
[DISCOVERY] Reference status: FOUND_IN_REFERENCE
[DISCOVERY] Explored domains: ['sanofi.com']
[DDGS] Search failed for Sanofi / assurance_report / 2023: ('error sending request for url (https://html.duckduckgo.com/html/)', 'https://html.duckduckgo.c

## 15. Analyse de couverture documentaire et collecte secondaire ciblée

### 1. Évaluation de la couverture documentaire prioritaire

Cette cellule mesure, pour chaque couple entreprise-année, la qualité de la collecte principale sur les documents ESG prioritaires.

On calcule notamment :
- le nombre de documents prioritaires sélectionnés automatiquement ;
- le nombre de documents nécessitant une revue manuelle ;
- le score moyen et maximal ;
- un ratio de couverture documentaire.

Ce diagnostic permet d’identifier les entreprises pour lesquelles la collecte principale est suffisante et celles qui nécessitent une collecte complémentaire.

In [51]:
# ============================================================
# 1. Score de couverture des documents importants
# ============================================================

CORE_REQUIRED_DOCS = [
    "annual_report_urd",
    "sustainability_statement_csrd_esrs",
    "climate_report_tcfd_transition_plan",
    "vigilance_plan",
    "assurance_report"
]


def build_core_coverage_score(selection_core, required_docs=CORE_REQUIRED_DOCS):
    df = selection_core.copy()

    df["is_selected"] = df["selection_status"].eq("SELECTED_FOR_DOWNLOAD")
    df["is_review"] = df["selection_status"].eq("REVIEW_REQUIRED")

    coverage = (
        df.groupby(["company", "fiscal_year"], dropna=False)
        .agg(
            n_core_docs=("doc_subtype", "count"),
            n_selected_core=("is_selected", "sum"),
            n_review_core=("is_review", "sum"),
            avg_core_score=("score", "mean"),
            max_core_score=("score", "max")
        )
        .reset_index()
    )

    coverage["core_coverage_ratio"] = (
        coverage["n_selected_core"] / len(required_docs)
    )

    def classify(row):
        if row["core_coverage_ratio"] >= 0.6 and row["avg_core_score"] >= 12:
            return "GOOD_CORE_COVERAGE"
        elif row["core_coverage_ratio"] >= 0.4:
            return "MEDIUM_CORE_COVERAGE"
        else:
            return "LOW_CORE_COVERAGE"

    coverage["core_coverage_status"] = coverage.apply(classify, axis=1)

    return coverage


core_coverage = build_core_coverage_score(selection_cac40_core)

display(
    core_coverage.sort_values(
        ["core_coverage_status", "core_coverage_ratio"],
        ascending=[True, False]
    )
)



,company,fiscal_year,n_core_docs,n_selected_core,n_review_core,avg_core_score,max_core_score,core_coverage_ratio,core_coverage_status
0,AXA,2023,5,5,0,29.6,37,1.0,GOOD_CORE_COVERAGE
1,AXA,2024,5,5,0,24.6,32,1.0,GOOD_CORE_COVERAGE
2,Accor,2023,5,5,0,27.6,35,1.0,GOOD_CORE_COVERAGE
3,Accor,2024,5,5,0,24.8,32,1.0,GOOD_CORE_COVERAGE
4,Air Liquide,2023,5,5,0,29.4,33,1.0,GOOD_CORE_COVERAGE
...,...,...,...,...,...,...,...,...,...
17,Capgemini,2024,5,4,1,20.6,29,0.8,GOOD_CORE_COVERAGE
43,LVMH,2024,5,4,1,21.8,29,0.8,GOOD_CORE_COVERAGE
57,STMicroelectronics,2024,5,4,1,21.8,29,0.8,GOOD_CORE_COVERAGE
50,Pernod Ricard,2023,5,3,2,20.2,29,0.6,GOOD_CORE_COVERAGE


### 2. Identification des besoins de collecte 

Cette cellule applique une règle de décision simple : la collecte secondaire est activée uniquement pour les couples entreprise-année dont la couverture documentaire principale est insuffisante ou moyenne.

L’objectif est d’éviter une collecte secondaire systématique sur tout l’univers CAC 40, afin de réduire :
- le temps de calcul ;
- les requêtes web inutiles ;
- le bruit documentaire ;
- les risques de doublons.

La collecte secondaire devient ainsi ciblée, conditionnelle et justifiée par un diagnostic de couverture.

In [52]:
def choose_secondary_scope(core_coverage):
    target_statuses = [
        "LOW_CORE_COVERAGE",
        "MEDIUM_CORE_COVERAGE"
    ]

    secondary_targets = (
        core_coverage[
            core_coverage["core_coverage_status"].isin(target_statuses)
        ]
        .copy()
        .sort_values(["core_coverage_status", "core_coverage_ratio"])
    )

    return secondary_targets


secondary_targets = choose_secondary_scope(core_coverage)

SECONDARY_TARGET_COMPANIES = sorted(secondary_targets["company"].dropna().unique())
SECONDARY_TARGET_YEARS = sorted(secondary_targets["fiscal_year"].dropna().unique())

print("Nombre de couples entreprise-année ciblés :", len(secondary_targets))
print("Nombre d'entreprises ciblées :", len(SECONDARY_TARGET_COMPANIES))
print("Années ciblées :", SECONDARY_TARGET_YEARS)

display(secondary_targets)

Nombre de couples entreprise-année ciblés : 1
Nombre d'entreprises ciblées : 1
Années ciblées : [np.int64(2023)]


,company,fiscal_year,n_core_docs,n_selected_core,n_review_core,avg_core_score,max_core_score,core_coverage_ratio,core_coverage_status
74,Unibail-Rodamco-Westfield,2023,5,1,4,14.6,29,0.2,LOW_CORE_COVERAGE


### 3. Collecte secondaire ciblée

Cette cellule lance une collecte complémentaire uniquement sur les entreprises et années identifiées comme insuffisamment couvertes lors du run principal.

Le registre secondaire contient des documents utiles mais moins prioritaires que les rapports principaux :
- politiques ESG ;
- codes de conduite ;
- documents fournisseurs ;
- présentations investisseurs ;
- réponses CDP ;
- validations SBTi ;
- rapports semestriels.

Cette étape permet d’enrichir la base documentaire sans alourdir inutilement la collecte globale.

In [53]:
if len(secondary_targets) == 0:
    print("Aucune collecte secondaire nécessaire : la couverture principale est suffisante.")

    discovery_cac40_secondary_targeted = pd.DataFrame(columns=DISCOVERY_COLUMNS)
    selection_cac40_secondary_targeted = pd.DataFrame(columns=SELECTION_COLUMNS)

else:
    discovery_cac40_secondary_targeted = run_document_discovery_optimized(
        companies=SECONDARY_TARGET_COMPANIES,
        years=SECONDARY_TARGET_YEARS,
        emitters_reference=EMITTERS,
        document_registry=SECONDARY_REPORTS_REGISTRY,
        allow_domain_guess=False
    )

    selection_cac40_secondary_targeted = select_best_documents(
        discovery_results=discovery_cac40_secondary_targeted,
        min_score_download=16,
        min_score_review=9
    )

    print("===== RUN SECONDAIRE CIBLÉ =====")
    print("Discovery:", discovery_cac40_secondary_targeted.shape)
    print("Selection:", selection_cac40_secondary_targeted.shape)

    display(
        selection_cac40_secondary_targeted[
            "selection_status"
        ].value_counts(dropna=False)
    )

    display(selection_cac40_secondary_targeted.head(50))


[DISCOVERY] Company: Unibail-Rodamco-Westfield
[DISCOVERY] Reference status: FOUND_IN_REFERENCE
[DISCOVERY] Explored domains: ['urw.com']
[DDGS] Search failed for Unibail-Rodamco-Westfield / environmental_policy / current: ('error sending request for url (https://html.duckduckgo.com/html/)', 'https://html.duckduckgo.com/html/')
[DISCOVERY] Unique PDF candidates found: 7
===== RUN SECONDAIRE CIBLÉ =====
Discovery: (55, 20)
Selection: (11, 17)


selection_status
REVIEW_REQUIRED          6
SELECTED_FOR_DOWNLOAD    5
Name: count, dtype: int64

,company,ticker,isin,jurisdiction,strate,doc_type,doc_subtype,fiscal_year,period_type,expected_query,selected_url,score,confidence_level,estimated_probability,selection_status,selection_reason,retrieval_date
0,Unibail-Rodamco-Westfield,URW,FR0013326246,France,3,corporate_communication,agm_minutes_resolutions,2023,adhoc,Unibail-Rodamco-Westfield annual general meeti...,https://www.bnains.org/archives/communiques/Un...,20,HIGH_CONFIDENCE,0.881,SELECTED_FOR_DOWNLOAD,Best candidate passed automatic download thres...,2026-05-10T08:40:05.687988+00:00
1,Unibail-Rodamco-Westfield,URW,FR0013326246,France,4,external_source,cdp_response,2023,FY,Unibail-Rodamco-Westfield CDP climate change r...,https://www.bnains.org/archives/communiques/Un...,21,HIGH_CONFIDENCE,0.900,SELECTED_FOR_DOWNLOAD,Best candidate passed automatic download thres...,2026-05-10T08:40:05.689835+00:00
2,Unibail-Rodamco-Westfield,URW,FR0013326246,France,1,regulatory,half_year_financial_report,2023,H1,Unibail-Rodamco-Westfield half year financial ...,https://www.bnains.org/archives/communiques/Un...,20,HIGH_CONFIDENCE,0.881,SELECTED_FOR_DOWNLOAD,Best candidate passed automatic download thres...,2026-05-10T08:40:05.691363+00:00
3,Unibail-Rodamco-Westfield,URW,FR0013326246,France,3,corporate_communication,investor_presentation,2023,adhoc,Unibail-Rodamco-Westfield investor presentatio...,https://www.bnains.org/archives/communiques/Un...,20,HIGH_CONFIDENCE,0.881,SELECTED_FOR_DOWNLOAD,Best candidate passed automatic download thres...,2026-05-10T08:40:05.693034+00:00
4,Unibail-Rodamco-Westfield,URW,FR0013326246,France,4,external_source,sbti_validation,2023,adhoc,Unibail-Rodamco-Westfield SBTi validated targets,https://www.bnains.org/archives/communiques/Un...,19,HIGH_CONFIDENCE,0.858,SELECTED_FOR_DOWNLOAD,Best candidate passed automatic download thres...,2026-05-10T08:40:05.694599+00:00
5,Unibail-Rodamco-Westfield,URW,FR0013326246,France,2,policy,anti_corruption_policy,current,perpetual,Unibail-Rodamco-Westfield anti corruption poli...,None,11,MEDIUM_CONFIDENCE,0.550,REVIEW_REQUIRED,Candidate exists but confidence is not suffici...,2026-05-10T08:40:05.696044+00:00
6,Unibail-Rodamco-Westfield,URW,FR0013326246,France,2,policy,code_of_conduct,current,perpetual,Unibail-Rodamco-Westfield code of conduct PDF,None,11,MEDIUM_CONFIDENCE,0.550,REVIEW_REQUIRED,Candidate exists but confidence is not suffici...,2026-05-10T08:40:05.700451+00:00
7,Unibail-Rodamco-Westfield,URW,FR0013326246,France,2,policy,dei_policy,current,perpetual,Unibail-Rodamco-Westfield diversity equity inc...,None,11,MEDIUM_CONFIDENCE,0.550,REVIEW_REQUIRED,Candidate exists but confidence is not suffici...,2026-05-10T08:40:05.703143+00:00
8,Unibail-Rodamco-Westfield,URW,FR0013326246,France,2,policy,environmental_policy,current,perpetual,Unibail-Rodamco-Westfield environmental policy...,None,11,MEDIUM_CONFIDENCE,0.550,REVIEW_REQUIRED,Candidate exists but confidence is not suffici...,2026-05-10T08:40:05.704863+00:00
9,Unibail-Rodamco-Westfield,URW,FR0013326246,France,2,policy,human_rights_policy,current,perpetual,Unibail-Rodamco-Westfield human rights policy PDF,None,12,MEDIUM_CONFIDENCE,0.599,REVIEW_REQUIRED,Candidate exists but confidence is not suffici...,2026-05-10T08:40:05.706535+00:00


### 4. Ingestion des documents secondaires sélectionnés

Cette cellule télécharge et enregistre uniquement les documents secondaires qui ont été retenus automatiquement après la phase de sélection.

Elle prolonge donc le run secondaire ciblé en appliquant la même logique d’ingestion que pour les documents prioritaires :
- téléchargement du PDF ;
- validation technique ;
- détection des doublons ;
- enregistrement des métadonnées ;
- mise à jour des logs de collecte.

Les documents nécessitant une revue manuelle ne sont pas ingérés automatiquement.

In [54]:
if selection_cac40_secondary_targeted.empty:
    print("Aucun document secondaire sélectionné : aucune ingestion secondaire lancée.")

    ingestion_cac40_secondary_selected = []

else:
    secondary_to_ingest = selection_cac40_secondary_targeted[
        selection_cac40_secondary_targeted["selection_status"].eq("SELECTED_FOR_DOWNLOAD")
    ].copy()

    ingestion_cac40_secondary_selected = ingest_company_documents_from_selection(
        selection_df=secondary_to_ingest,
        emitters_reference=EMITTERS
    )

    print("===== INGESTION SECONDAIRE SÉLECTIVE =====")
    print(
        "Documents secondaires effectivement ingérés :",
        len([x for x in ingestion_cac40_secondary_selected if x is not None])
    )

===== INGESTION SECONDAIRE SÉLECTIVE =====
Documents secondaires effectivement ingérés : 1
